# Module 14 — PriceMind AI Full Platform Integration

This notebook demonstrates the end-to-end integration of all 14 PriceMind AI modules:
- **Authentication & Multi-Tenancy** (JWT, Organizations)
- **Product Catalog & Consolidated Analytics** (`/products/{id}/analytics`)
- **Price Elasticity & Demand Forecasting** (OLS + Prophet/Spline)
- **Constraint-Bounded Optimization** (`/pricing/recommend`)
- **TreeSHAP Explainability** (`/explanations/pricing`)
- **LangChain RAG Knowledge System** (`/rag/query`)
- **LangChain AI Pricing Agent** (`/agent/query`)


In [ ]:
import sys
import json
from pathlib import Path

# Setup python paths
ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'backend'))
print('Root directory:', ROOT)


## 1. Authentication & Organization Tenant Context


In [ ]:
from fastapi.testclient import TestClient
from app.main import app

client = TestClient(app)

reg_res = client.post('/api/v1/auth/register', json={
    'full_name': 'Dr. Elena Rostova',
    'email': 'elena.rostova@enterprise.com',
    'organization_name': 'Global Pricing Corp',
    'password': 'SecurePassword2026!',
    'confirm_password': 'SecurePassword2026!',
})

if reg_res.status_code == 201:
    token = reg_res.json()['access_token']
    print('Authenticated as:', reg_res.json()['user']['email'])
else:
    login_res = client.post('/api/v1/auth/login', json={
        'email': 'elena.rostova@enterprise.com',
        'password': 'SecurePassword2026!'
    })
    token = login_res.json()['access_token']
    print('Logged in successfully')

headers = {'Authorization': f'Bearer {token}'}


## 2. Product Catalog & Consolidated Analytics (`/products/{id}/analytics`)


In [ ]:
products_res = client.get('/api/v1/products')
products = products_res.json()
first_sku = products[0]['skuCode']
print(f'Retrieved {len(products)} products. Analyzing first product: {first_sku}')

analytics_res = client.get(f'/api/v1/products/{first_sku}/analytics')
print('\nConsolidated Product Analytics:')
print(json.dumps(analytics_res.json(), indent=2))


## 3. End-to-End Pricing Recommendation Pipeline (`/pricing/recommend`)


In [ ]:
rec_res = client.post('/api/v1/pricing/recommend', json={
    'product_id': first_sku,
    'objective': 'PROFIT_MAX'
})
rec_data = rec_res.json()
print('Recommended Price:', rec_data.get('recommended_price'))
print('Expected Profit:', rec_data.get('expected_profit'))
print('Elasticity:', rec_data.get('elasticity'), f'({rec_data.get("elasticity_category")})')
print('SHAP Base Value:', rec_data.get('explanation', {}).get('base_value'))


## 4. Multi-Tool AI Pricing Agent (`/agent/query`)


In [ ]:
agent_res = client.post(
    '/api/v1/agent/query',
    json={'message': f'Why should {first_sku} optimize its pricing?'},
    headers=headers,
)
agent_data = agent_res.json()
print('Agent Answer:')
print(agent_data.get('answer'))
print('\nTools Used:', agent_data.get('tools_used'))
print('Sources:', [s['title'] for s in agent_data.get('sources', [])])
